# Equal-Weight Backtest - FX Pairs

This notebook establishes the strategy baseline. At each decision time it ranks the currency-pair
scores, takes equal-sized long and short sleeves, and runs the existing FX backtest engine. Every
complete model configuration and checkpoint is evaluated. The input is a Polars catalog selection,
so no prediction hash is copied into orchestration code.

This is the step that turns a model into a strategy. Everything before it scored pairs; nothing
before it has said what you would have held or what it would have returned. Rank correlation
and AUC measure whether the ordering carries information, and a model can be clearly informative
by both and still lose money once the ordering is turned into positions - the information may
sit in pairs too illiquid to hold, or in a part of the distribution the sleeves never reach.
The return series produced here is the first quantity in the chain that a trader could have
experienced, and it is the reference every later stage is measured against.

Equal weight is used deliberately rather than for convenience. It is the choice not to choose a
size, so the resulting return series carries the ranking's contribution and nothing else. Any
other weighting would blend the model's information with a view about sizing, and the
allocation notebook could then no longer attribute its improvement to allocation.


<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 等權重回測——外匯貨幣對

本 Notebook 建立策略基準。在每個決策時間，它依貨幣對分數排序，建立等大小的多頭與空頭部位組，並使用既有外匯回測引擎執行。每個完整的模型配置與檢查點都會接受評估。輸入來自 Polars 目錄篩選，因此無須將預測雜湊值複製到流程協調程式碼中。

這一步將模型轉換為策略。此前所有步驟都在替貨幣對評分；此前沒有任何步驟說明實際會持有哪些部位，或會取得多少報酬。等級相關性與 AUC 衡量排序是否含有資訊；即使兩者都明確顯示模型具有資訊，排序轉為部位後仍可能虧損——資訊可能集中在流動性太低、無法持有的貨幣對，或集中在多空部位組永遠觸及不到的分布區段。此處產生的報酬率序列，是整個流程中交易者第一個可能實際經歷的量，也是衡量後續每一階段的參考基準。

採用等權重是刻意的選擇，不只是為了方便。它代表不另行選擇部位大小，因此產生的報酬率序列只包含排序的貢獻。其他加權方式都會把模型資訊與部位規模觀點混合，屆時資產配置 Notebook 便無法將改善歸因於配置。
<!-- ML4T_ZH_END c0000-p000 -->

**A note on how this appears in the registry.** These runs register with `stage='signal'`, and
the notebook asserts it. That stored value names the strategy component being varied - here the
signal mapping - and it does not mean there is a separate "signal stage" upstream of a backtest.
These *are* the baseline backtests. In prose, write "the baseline backtests" and "baseline
(equal-weight) Sharpe", never "signal Sharpe", which reads as a metric of the predictions rather
than of a traded strategy.

**Learning objectives**

- Freeze the exact prediction population before running a baseline sweep.
- Send selected catalog rows to the canonical FX engine.
- Verify that every declared prediction and portfolio-size combination produces one backtest.
- Read the baseline as the reference the allocation, risk and cost stages are measured against.

**Book reference**: Chapter 16

**Prerequisite**: `12_model_analysis`.

<!-- ML4T_ZH_BEGIN c0000-p001 -->
**關於登錄系統呈現方式的說明。**這些執行以 `stage='signal'` 登記，Notebook 也會對此進行斷言。這個儲存值標示正在變動的策略組件——此處是訊號映射——並不表示回測上游另有一個獨立的「訊號階段」。這些*就是*基準回測。在文字敘述中，應寫成「基準回測」與「基準（等權重）夏普比率」，絕不寫成「訊號夏普比率」；後者會被理解為預測值的指標，而不是已交易策略的指標。

**學習目標**

- 在執行基準參數掃描前，固定確切的預測模型集合。
- 將選定的目錄資料列送入標準外匯引擎。
- 驗證每個已宣告的預測與投資組合規模組合，都會產生一次回測。
- 將基準結果作為衡量配置、風險與成本階段的參考。

**書籍參考**：第 16 章

**先備條件**：`12_model_analysis`。
<!-- ML4T_ZH_END c0000-p001 -->


In [1]:
"""Run the complete equal-weight FX validation backtest population."""

import polars as pl
import yaml

from case_studies.research import (
    OfficialPopulation,
    open_study,
    plan_backtests,
    population_supersedes,
    research_name,
    reuse_disclosure,
    run_backtests,
    superseded_members,
)
from case_studies.utils.sweep_config import get_top_k_values_for
from utils.paths import get_case_study_dir
from utils.reproducibility import set_global_seeds

In [2]:
CASE_STUDY_ID = "fx_pairs"
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
LABEL = ""
SPLIT = "validation"
TOP_K = 0
SEED = 42
RUN_SWEEP = True
FORCE_REBACKTEST = False
TOP_N_PREDICTIONS = None
POPULATION_NAME = ""
# Both name the lineage rather than a generation of it. The hashes these replace -
# `59edb30b6271` and `16787d96bf75` - are in no lineage the registry holds, and the names
# they publish under are built by `research_name`, so nothing could look them up to say
# whether they were dead or waiting for a first publication. `"live"` does not have to
# decide: it resolves against the name this run publishes under, which is the one thing
# both call sites already know. See `case_studies.research.population.SUPERSEDES_LIVE`.
SUPERSEDES_EQUAL_WEIGHT_BASELINES: str = "live"
SUPERSEDES_VALIDATION_PREDICTIONS: str = "live"

## Select the exact prediction population

Canonical production includes every complete validation prediction the model notebooks currently
publish. Label, configuration, or population limits are preview controls and cannot define the
official baseline.

**"Currently publish" is a question about lineage, and the catalog cannot answer it.** A row's
`identity_status` is derived from the schema version it was written under, so it says the registry
still understands the row - not that the row is the one its producer stands behind. The two agree
until a model notebook refits. Then it publishes a second generation of its population under the
same name, the first generation's prediction sets stay in the registry complete and current, and a
sweep selecting on the catalog alone runs over both. It would not fail; it would report every
member complete, over twice the population, and freeze the retired half into the baseline the rest
of the case study is measured against.

`superseded_members` asks the registry which identities a later generation retired and no
generation in force still lists, which is exactly the set to drop. The `tabular_dl` and
`deep_learning` populations each have a retired generation here, because a training identity
covers the runner's own source file and both runners changed after their first fits were
registered. The count of what that excludes is printed rather than left implicit.


<!-- ML4T_ZH_BEGIN c0003-p000 -->
## 選擇確切的預測模型集合

標準正式執行包含模型 Notebook 目前發布的每項完整驗證預測。標籤、配置或模型集合數量限制都屬於預覽控制，不能定義正式基準。

**「目前發布」涉及版本沿革，而目錄無法回答這個問題。**資料列的 `identity_status` 是由寫入時的架構版本推導而來，因此它只表示登錄系統仍能理解該資料列，並不表示產生端目前仍認可它。兩者在模型 Notebook 重新擬合前一致。重新擬合後，模型會在同一名稱下發布其集合的第二代；第一代預測集仍以完整且目前架構可讀的狀態留在登錄系統，而只按目錄選取的參數掃描會同時涵蓋兩代。這不會失敗；它會回報所有成員皆完整，但模型集合數量超過兩倍，並把已退役的一半固定到後續整個案例研究用來衡量的基準中。

`superseded_members` 會詢問登錄系統：哪些識別資訊已被後續版本退役，且不再列於任何生效版本；這正是應排除的集合。`tabular_dl` 與 `deep_learning` 模型集合在此各有一代已退役，因為訓練識別資訊涵蓋執行器自身的來源檔案，而兩個執行器都在首次擬合結果登記後發生變更。排除數量會明確列印，不會隱藏。
<!-- ML4T_ZH_END c0003-p000 -->

**A published population can need a second generation too.** The two names this notebook
publishes are lists of identities, and the exclusion above changes both of them the moment a
model notebook refits: the prediction population loses the retired members, and every baseline
backtest resolved from them goes with it. `OfficialPopulation.create` refuses a changed list
under an existing name without being told which snapshot it replaces, so each name has its own
declaration and each is offered through `population_supersedes` on the same rule the model
notebooks use. Both are empty here because neither name has published a first generation yet;
after the first canonical run, a refit upstream is answered by filling in the hash that run
printed.

<!-- ML4T_ZH_BEGIN c0003-p001 -->
**已發布的模型集合也可能需要第二代。**本 Notebook 發布的兩個名稱各代表一份識別資訊清單，而上述排除會在模型 Notebook 重新擬合時立即改變這兩份清單：預測模型集合會失去已退役成員，由它們解析出的每項基準回測也會跟著移除。若未告知 `OfficialPopulation.create` 新清單取代哪個快照，它便會拒絕在既有名稱下寫入變更後的清單，因此每個名稱都有自己的宣告，並依模型 Notebook 所用的相同規則，透過 `population_supersedes` 提交。此處兩者皆為空，因為兩個名稱都還未發布第一代；首次標準執行後，若上游重新擬合，就填入該次執行所列印的雜湊值來處理。
<!-- ML4T_ZH_END c0003-p001 -->


In [3]:
set_global_seeds(SEED)
if SPLIT != "validation":
    raise ValueError("the baseline sweep uses validation predictions")
if FORCE_REBACKTEST:
    raise ValueError("identical complete backtests are reused by identity")
if not RUN_SWEEP:
    raise ValueError("set RUN_SWEEP=True to execute the visible baseline request")

study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
# The execution tier decides which registry namespace this run reads and writes;
# the reduction knobs decide only how much of it is covered. Inferring the tier
# from the knobs conflated the two, so any reduced run went looking for preview
# predictions - and a reduced run over a canonical upstream, which is what the
# test suite exercises, then resolved no rows at all.
include_preview = EXECUTION_TIER == "preview"
catalog = study.predictions.table(include_preview=include_preview).filter(
    (pl.col("identity_status") == "current") & (pl.col("split") == SPLIT) & pl.col("complete")
)
if include_preview:
    catalog = catalog.filter(pl.col("execution_tier") == "preview")
else:
    catalog = catalog.filter(pl.col("execution_tier") == "canonical")
retired = superseded_members(study, member_kind="prediction")
if retired:
    offered = catalog.height
    catalog = catalog.filter(~pl.col("prediction_hash").is_in(list(retired)))
    print(f"Retired by a later generation, excluded: {offered - catalog.height} of {offered}")
if LABEL:
    catalog = catalog.filter(pl.col("label") == LABEL)
if TOP_N_PREDICTIONS is not None:
    catalog = catalog.sort("label", "family", "config_name", "checkpoint_value").head(
        TOP_N_PREDICTIONS
    )

if catalog.is_empty():
    raise RuntimeError("the baseline request resolved no complete prediction rows")

# TOP_K and TOP_N_PREDICTIONS both narrow what is backtested, and a narrowed run declares
# a different set of members than the canonical population does. A population is immutable
# once written, so such a run must publish under its own name rather than register a
# partial snapshot under the canonical one. The tier is a separate question: a canonical
# run may legitimately be narrowed, it just may not claim to be the whole population.
if (
    (TOP_K or TOP_N_PREDICTIONS is not None or LABEL)
    and not include_preview
    and not POPULATION_NAME
):
    raise ValueError(
        "this run narrows the baseline sweep, so it cannot publish the canonical "
        "population; pass POPULATION_NAME to give it its own"
    )
if catalog.get_column("prediction_hash").n_unique() != catalog.height:
    raise RuntimeError("the baseline population contains duplicate prediction identities")

if not include_preview:
    predictions_name = research_name(CASE_STUDY_ID, "validation-predictions", scope=POPULATION_NAME)
    prediction_population = OfficialPopulation.create(
        study,
        name=predictions_name,
        member_kind="prediction",
        members=catalog.get_column("prediction_hash").to_list(),
        supersedes=population_supersedes(
            study, name=predictions_name, declared=SUPERSEDES_VALIDATION_PREDICTIONS
        ),
    )
    prediction_population.require_complete()
    print(f"Frozen prediction population: {prediction_population.hash}")
else:
    print("Preview selection is isolated from the official prediction population.")

catalog.select(
    "label",
    "family",
    "config_name",
    "checkpoint_kind",
    "checkpoint_value",
    "prediction_hash",
)

Retired by a later generation, excluded: 180 of 966


Frozen prediction population: a7d03d47eb24


label,family,config_name,checkpoint_kind,checkpoint_value,prediction_hash
str,str,str,str,i64,str
"""fwd_ret_1d""","""deep_learning""","""nlinear""","""epoch""",20,"""0016f5cad4b6"""
"""fwd_ret_21d""","""linear""","""ridge_a100000.0""","""final""",null,"""001742474fa3"""
"""fwd_ret_5d""","""deep_learning""","""lstm_h64""","""epoch""",45,"""00877e7b6a7a"""
"""fwd_ret_5d""","""deep_learning""","""nlinear""","""epoch""",20,"""008b93c8017c"""
"""fwd_ret_21d""","""gbm""","""leaves_31_huber""","""iteration""",50,"""00fc9b95d65b"""
…,…,…,…,…,…
"""fwd_ret_1d""","""gbm""","""leaves_31_huber""","""iteration""",150,"""ff0da2db9b9c"""
"""fwd_ret_1d""","""deep_learning""","""nlinear""","""epoch""",100,"""ff810fb5104f"""
"""fwd_ret_5d""","""deep_learning""","""nlinear""","""epoch""",45,"""ff8c4352ccb4"""


## Build the baseline strategy grid

`top_k` is the number of pairs in each sleeve. This account allows short selling, so the engine
holds `top_k` long and `top_k` short and clamps each sleeve to half the universe to keep them
disjoint. Two consequences worth stating exactly, because both shape how the grid reads:

- At `top_k` below half the universe, ranking decides *which* pairs are held.
- At `top_k` equal to half the universe, every pair is held at every rebalance and ranking decides
  only the *side*. The grid then varies concentration and sign exposure, not membership.

A `top_k` above half the universe cannot express anything the half-universe value does not: the
engine clamps it to the same sleeves, so it would register a distinct identity carrying an
identical weight series. The grid is refused rather than allowed to contain that duplicate.

<!-- ML4T_ZH_BEGIN c0005-p000 -->
## 建立基準策略網格

`top_k` 是每個多空部位組中的貨幣對數。本帳戶允許放空，因此引擎持有 `top_k` 個多頭與 `top_k` 個空頭，並將每組的大小限制在交易標的集合的一半，以確保兩組不重疊。以下兩個結果值得明確說明，因為它們都影響網格的解讀：

- 當 `top_k` 小於交易標的集合的一半時，排序決定持有*哪些*貨幣對。
- 當 `top_k` 等於交易標的集合的一半時，每次再平衡都會持有所有貨幣對，排序只決定*多空方向*。此時網格變動的是集中度與方向曝險，而非成員組成。

若 `top_k` 大於交易標的集合的一半，便無法表達超出半數設定所能表達的內容：引擎會將其限制為相同的部位組，因此會登記不同的識別資訊，卻得到相同的權重序列。網格會拒絕這種重複項目，而非容許它存在。
<!-- ML4T_ZH_END c0005-p000 -->


In [4]:
universe_symbols = yaml.safe_load(
    (get_case_study_dir(CASE_STUDY_ID) / "config" / "setup.yaml").read_text()
)["universe"]["symbols"]
n_assets = len(universe_symbols)
max_sleeve = n_assets // 2

jobs = []
for label in sorted(catalog.get_column("label").unique()):
    selected = catalog.filter(pl.col("label") == label)
    top_k_values = [TOP_K] if TOP_K else get_top_k_values_for(CASE_STUDY_ID, label, n_assets)
    duplicates = sorted({value for value in top_k_values if value > max_sleeve})
    if duplicates:
        raise RuntimeError(
            f"top_k {duplicates} exceed the {max_sleeve}-pair sleeve ceiling for a long-short "
            f"account on {n_assets} pairs; the engine would clamp them to {max_sleeve} and register "
            "a duplicate weight series under a distinct identity"
        )
    for top_k in top_k_values:
        jobs.append(
            {
                "label": label,
                "top_k": top_k,
                "predictions": selected,
                "expected": selected.height,
            }
        )

pl.DataFrame(
    [
        {"label": job["label"], "top_k": job["top_k"], "prediction_sets": job["expected"]}
        for job in jobs
    ]
)

label,top_k,prediction_sets
str,i64,i64
"""fwd_ret_1d""",5,262
"""fwd_ret_1d""",10,262
"""fwd_ret_21d""",5,262
"""fwd_ret_21d""",10,262
"""fwd_ret_5d""",5,262
"""fwd_ret_5d""",10,262


## Freeze the expected baseline population

Planning resolves every backtest identity without running or writing it. Production freezes that
complete expected set before the first member executes, so a failed member remains visible.

Visibility is the whole point, and it is worth being explicit about the alternative. A sweep that
collected whatever finished would publish a baseline whose membership depends on which runs
happened to succeed. Failures are not random with respect to the thing being measured: a
configuration that produces degenerate scores, or one whose predictions are too sparse to fill a
sleeve, is more likely to fail and would silently leave the population. The surviving baseline
would then look better than the sweep that produced it, and nothing in the output would record
that anything was dropped. Declaring the expected set first converts that from a silent
improvement into an incomplete population that refuses to publish.

<!-- ML4T_ZH_BEGIN c0007-p000 -->
## 固定預期的基準模型集合

規劃程序會解析每項回測的識別資訊，但不執行或寫入回測。正式執行會在第一個成員開始前，固定完整的預期集合，讓失敗的成員仍可見。

可見性正是重點，因此值得明確說明另一種作法的問題。若參數掃描只收集已完成的結果，發布的基準成員組成就會取決於哪些執行碰巧成功。失敗與被衡量的事物並非毫無關係：若某配置產生退化的分數，或其預測過於稀疏、無法填滿部位組，就較容易失敗，並悄悄退出集合。倖存的基準看起來會比產生它的參數掃描更好，而輸出卻不會記錄有任何項目遭到排除。先宣告預期集合，便能將這種無聲的改善轉為拒絕發布的不完整模型集合。
<!-- ML4T_ZH_END c0007-p000 -->


In [5]:
planned_hashes = []
for job in jobs:
    plan = plan_backtests(
        study,
        predictions=job["predictions"],
        signal={"method": "equal_weight_top_k", "top_k": job["top_k"]},
        chapter="16",
    )
    if len(plan.members) != job["expected"]:
        raise RuntimeError("a baseline plan omitted a selected prediction")
    planned_hashes.extend(plan.expected_hashes)

if len(planned_hashes) != len(set(planned_hashes)):
    raise RuntimeError("two planned baseline jobs collapse to the same backtest identity")

baseline_population = None
if not include_preview:
    baselines_name = research_name(CASE_STUDY_ID, "equal-weight-baselines", scope=POPULATION_NAME)
    baseline_population = OfficialPopulation.create(
        study,
        name=baselines_name,
        member_kind="backtest",
        members=planned_hashes,
        supersedes=population_supersedes(
            study, name=baselines_name, declared=SUPERSEDES_EQUAL_WEIGHT_BASELINES
        ),
    )
    print(f"Frozen expected baseline population: {baseline_population.hash}")
else:
    print("Preview backtests remain outside official populations and selection.")

Frozen expected baseline population: cbf9699606a3


## Run every catalog row through the FX engine, then validate what was frozen

Each selected row produces an independent backtest. The loop has no exception-and-continue path:
one failed member leaves the predeclared population incomplete and stops publication.

The population is validated in the same cell that fills it, because the two are one act: the
expected set was written down before the first member ran, and `require_complete` is the only
thing that turns it from a declaration into a published result. It can pass only when every
planned model, checkpoint and portfolio size completed.

<!-- ML4T_ZH_BEGIN c0009-p000 -->
## 讓每個目錄資料列都通過外匯引擎，再驗證已固定的集合

每個選定資料列都會產生獨立回測。迴圈沒有捕捉例外後繼續執行的路徑：只要一個成員失敗，就會使預先宣告的集合不完整，並停止發布。

模型集合會在填入成員的同一儲存格中驗證，因為這兩者屬於同一動作：預期集合早在第一個成員執行前就已記錄，而 `require_complete` 是唯一能使它從宣告變成已發布結果的步驟。唯有所有規劃中的模型、檢查點與投資組合規模都完成，驗證才會通過。
<!-- ML4T_ZH_END c0009-p000 -->


In [6]:
# A sweep that recomputes everything and a sweep that recomputes nothing print the same summary
# unless the two are counted apart. `run_backtests` serves an identity that is already registered
# and complete instead of running it again, which is what makes a re-run affordable and what makes
# a bare member count say nothing about whether this run did any work.
#
# The runner already knows which it did and says so per member in `execution.diagnostics`, as
# `status` "reused" or "completed". Comparing against the registered hashes instead would be
# wrong in both directions: a registered-but-partial backtest is in that set, gets recomputed and
# would report as reused, and a preview re-run reads a table that excludes preview rows by default
# and would report every reused member as computed.
run_status: list[str] = []
backtests = []
for job in jobs:
    execution = run_backtests(
        study,
        predictions=job["predictions"],
        signal={"method": "equal_weight_top_k", "top_k": job["top_k"]},
        chapter="16",
    )
    if len(execution.results) != job["expected"]:
        raise RuntimeError("a baseline member disappeared during execution")
    backtests.extend(execution.results)
    run_status.extend(entry["status"] for entry in execution.diagnostics)

expected_count = sum(job["expected"] for job in jobs)
if len(backtests) != expected_count:
    raise RuntimeError(f"expected {expected_count} baseline runs, found {len(backtests)}")
if {result.hash for result in backtests} != set(planned_hashes):
    raise RuntimeError("completed baseline identities differ from the frozen plan")
if any(not result.complete for result in backtests):
    raise RuntimeError("the baseline population contains an incomplete backtest")

backtest_rows = pl.DataFrame(
    [
        {
            "backtest_hash": result.hash,
            "prediction_hash": result.registry_record()["prediction_hash"],
            "stage": result.registry_record()["stage"],
            "complete": result.complete,
        }
        for result in backtests
    ]
)
if set(backtest_rows.get_column("stage")) != {"signal"}:
    raise RuntimeError("equal-weight baseline runs must register with stage='signal'")

served = run_status.count("reused")
print(
    f"Equal-weight baselines: {reuse_disclosure(len(backtests) - served, served)}, "
    f"{len(backtests)} in the population"
)

if not include_preview:
    if baseline_population is None:
        raise RuntimeError("the canonical baseline population was not frozen before execution")
    baseline_population.require_complete()
    print(f"Official equal-weight population: {baseline_population.hash}")

backtest_rows

Equal-weight baselines: computed 360, reused 1212 from the registry, 1572 in the population


Official equal-weight population: cbf9699606a3


backtest_hash,prediction_hash,stage,complete
str,str,str,bool
"""580a44618b66""","""0016f5cad4b6""","""signal""",true
"""42736ca1f61c""","""035323653d87""","""signal""",true
"""dc271c8e6761""","""03c835ba04d0""","""signal""",true
"""c7f777557420""","""045f1259f65a""","""signal""",true
"""95f25de7b47a""","""04956220e994""","""signal""",true
…,…,…,…
"""3b15729bd4a9""","""fccb33396cba""","""signal""",true
"""5d9e2677eb3b""","""fd559fb7b792""","""signal""",true
"""1b5590dc7274""","""fd6910b035a6""","""signal""",true


## Key takeaways

- The baseline evaluates every complete model configuration and checkpoint, so the population
  is configurations times checkpoints times portfolio sizes. That product is the number of
  trials every later comparison is drawn from.
- Equal weight isolates the ranking's contribution. It is the reference the allocation, risk and
  cost stages are each measured against, which is why it is computed before any of them.
- Catalog rows pass directly to the existing FX backtest engine.
- Immutable population membership makes missing or failed jobs visible. Failures correlate with
  the thing being measured, so a sweep that published its survivors would flatter itself.
- These rows carry `stage='signal'`. They are the baseline backtests; the stored value names the
  varied component, not a stage before the backtest.

The Sharpe ratios here are validation numbers over a single history, and the best of a large
grid is high partly because the grid is large. Nothing at this stage separates a configuration
that ranks well from one that ranked well once. That separation is what the holdout is for, and
it is spent once, at the end, against the whole candidate set rather than against the leader.

<!-- ML4T_ZH_BEGIN c0011-p000 -->
## 重點摘要

- 基準會評估每個完整的模型配置與檢查點，因此模型集合的大小等於配置數乘以檢查點數，再乘以投資組合規模數。這個乘積是所有後續比較所依據的試驗次數。
- 等權重單獨呈現排序的貢獻。它是分別衡量配置、風險與成本階段的參考基準，因此會先於這些階段計算。
- 目錄資料列直接送入既有外匯回測引擎。
- 不可變的模型集合成員組成，讓缺少或失敗的工作可見。失敗與被衡量的事物相關，因此只發布倖存項目的參數掃描，會高估自身表現。
- 這些資料列帶有 `stage='signal'`。它們是基準回測；該儲存值標示變動的組件，而不是回測前的階段。

此處的夏普比率，是單一歷史期間上的驗證期數值；大型網格的最佳值偏高，部分原因就是網格本身很大。在這個階段，仍無法分辨某配置是穩定地排序良好，還是只在某一次排序良好。保留樣本正是為了區分兩者，而且只會在最後使用一次，用來對照整個候選集合，而非只對照領先者。
<!-- ML4T_ZH_END c0011-p000 -->
